# Expected Values and Probabilities

Every expected value is an integral: $E[g(X)] = \int g(x)\, f(x)\, dx$, where $f$ is the density of $X$. Probabilities, moments, expected utility and prices of risky assets are all special cases. This notebook computes them by quadrature, checks them against closed forms, and compares quadrature with Monte Carlo.

In [ ]:
Pollis.packages("QuadGK", "FastGaussQuadrature", "Distributions", "Plots")

In [ ]:
using QuadGK, FastGaussQuadrature, Distributions, Random, Statistics, Plots
Random.seed!(1)

## Densities integrate to one, tails are integrals

A density must integrate to one over its support. The probability of an event is the integral of the density over that event, so a tail probability is an integral to infinity.

In [ ]:
X = Normal(0, 1)
(total = quadgk(x -> pdf(X, x), -Inf, Inf)[1],
 upper_tail_2 = quadgk(x -> pdf(X, x), 2, Inf)[1], closed_form = ccdf(X, 2),
 within_1_96 = quadgk(x -> pdf(X, x), -1.96, 1.96)[1])

## Moments

The mean, variance and skewness of $X \sim \text{LogNormal}(0, 0.5)$ come from integrals of $x$, $(x - \mu)^2$ and $(x - \mu)^3$ against the density. Distributions.jl knows the closed forms, so we can check.

In [ ]:
Y = LogNormal(0, 0.5)
E(g) = quadgk(x -> g(x) * pdf(Y, x), 0, Inf)[1]
m = E(x -> x)
v = E(x -> (x - m)^2)
s = E(x -> (x - m)^3) / v^1.5
[(moment = "mean", quadrature = m, closed_form = mean(Y)),
 (moment = "variance", quadrature = v, closed_form = var(Y)),
 (moment = "skewness", quadrature = s, closed_form = skewness(Y))]

## Expected value from the survival function

For a non-negative random variable, $E[X] = \int_0^\infty P(X > x)\, dx$. This form is used for lifetimes, claim sizes and waiting times, where the survival curve is often what is observed.

In [ ]:
W = Weibull(1.5, 10.0)   # shape 1.5, scale 10: a typical time-to-failure model
(from_density = quadgk(x -> x * pdf(W, x), 0, Inf)[1], from_survival = quadgk(x -> ccdf(W, x), 0, Inf)[1], closed_form = mean(W))

## Expected utility and the certainty equivalent

An agent with utility $u$ facing the risky payoff $Y$ values it at $E[u(Y)]$. The certainty equivalent is the sure amount with the same utility, $u^{-1}(E[u(Y)])$. The gap to $E[Y]$ is the risk premium.

In [ ]:
utilities = [("log", log, exp), ("sqrt", sqrt, x -> x^2), ("CRRA gamma = 3", x -> -x^(-2) / 2, u -> (-2u)^(-1 / 2))]
[(utility = name, expected_utility = E(u), certainty_equivalent = uinv(E(u)), risk_premium = mean(Y) - uinv(E(u))) for (name, u, uinv) in utilities]

## Gauss-Hermite: expectations of normal variables

When $X \sim N(\mu, \sigma^2)$, substituting $x = \mu + \sqrt{2}\,\sigma t$ gives

$$E[g(X)] = \frac{1}{\sqrt{\pi}} \int_{-\infty}^{\infty} g(\mu + \sqrt{2}\,\sigma t)\, e^{-t^2}\, dt \approx \frac{1}{\sqrt{\pi}} \sum_i w_i\, g(\mu + \sqrt{2}\,\sigma t_i).$$

A handful of Gauss-Hermite nodes is often exact to machine precision. This is the workhorse for expectations in dynamic economic models and for integrating out random effects in mixed models.

In [ ]:
function gh_expect(g, mu, sigma, n)
    t, w = gausshermite(n)
    return sum(w .* g.(mu .+ sqrt(2) * sigma .* t)) / sqrt(pi)
end
mu, sigma = 0.1, 0.3
g(x) = log(1 + exp(x))   # softplus: no closed form for its expectation
reference = quadgk(x -> g(x) * pdf(Normal(mu, sigma), x), -Inf, Inf; rtol=1e-12)[1]
[(nodes = n, gauss_hermite = gh_expect(g, mu, sigma, n), error = gh_expect(g, mu, sigma, n) - reference) for n in (2, 3, 5, 10, 20)]

## Quadrature or Monte Carlo?

Monte Carlo estimates $E[g(X)]$ by the sample mean of $g$ at random draws. Its error falls like $1/\sqrt{N}$ whatever the dimension. In one dimension, quadrature is far more accurate for the same number of evaluations.

In [ ]:
call(x) = max(x - 1.2, 0)   # a kinked payoff
exact_call = E(call)
[(N = N, monte_carlo_error = abs(mean(call.(rand(Y, N))) - exact_call)) for N in (10^2, 10^4, 10^6)]

In [ ]:
xs = range(0.01, 4, length=400)
plot(xs, pdf.(Y, xs), lw=2, label="density of Y", xlabel="y")
plot!(xs, call.(xs) .* pdf.(Y, xs), fillrange=0, fillalpha=0.3, label="payoff times density: its area is E[max(Y - 1.2, 0)]")

## Exercises

1. Compute $P(Y > 2)$ and the 95% quantile of $Y$ by integrating the density. Find the quantile with a root finder, then check against `quantile(Y, 0.95)`.
2. Use Gauss-Hermite to compute $E[e^{X}]$ for $X \sim N(\mu, \sigma^2)$ and compare with $e^{\mu + \sigma^2 / 2}$. How many nodes do you need?
3. For the CRRA utility with $\gamma = 1, 2, 5, 10$, plot the risk premium against $\gamma$.